<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 30px; border-radius: 15px; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; text-align: center; box-shadow: 0 10px 20px rgba(0,0,0,0.19), 0 6px 6px rgba(0,0,0,0.23);">
    <div style="font-size: 50px; margin-bottom: 10px;"> 📖</div>
    <h1 style="margin: 0; font-size: 36px; text-shadow: 2px 2px 4px rgba(0,0,0,0.3);">
        Tox21: Exploration Data Analisys
    </h1>
    <p style="font-size: 18px; margin-top: 5px; font-weight: 300;">
        Exploring data distributions, class imbalance, and missing values
    </p>
</div>

# Imports
---

In [ ]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
from itables import show
import seaborn as sns
import matplotlib.pyplot as plt
from rdkit import Chem
from config import DATASET_PATH
import numpy as np
import py3Dmol
import plotly.express as px
from rdkit.Chem import Descriptors,AllChem
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D
#%matplotlib widget

# Tox21: overview
---
it contains qualitative toxicity measurements on 12 biological targets, including nuclear receptors and stress response pathways.
For each molecule we will divide into 12 biological test characterized with two possible values (0/1):
 ### 1)  Group NR (Nuclear Receptors): Interferenti Endocrini 
- **NR-AR**		Recettore degli Androgeni (pieno)
- **NR-AR-LBD**		Recettore degli Androgeni (dominio legante)
- **NR-AhR**		Recettore degli Idrocarburi Arilici
- **NR-Aromatase**		Inibizione dell'Aromatasi
- **NR-ER**		Recettore degli Estrogeni (pieno)
- **NR-ER-LBD**		Recettore degli Estrogeni (dominio legante)
- **NR-PPAR-gamma**		Recettore nucleare PPAR-gamma
 ###  2) Group SR (Stress Response): Danno Cellulare
- **SR-ARE**	SR	Risposta antiossidante (stress ossidativo)
- **SR-ATAD5**	SR	Genotossicità (induzione di ATAD5)
- **SR-HSE**	SR	Risposta allo shock termico
- **SR-MMP**	SR	Potenziale di membrana mitocondriale
- **SR-p53**	SR	Attivazione della proteina p53 (danni al DNA)

In [ ]:
file_path = "tox21.csv"
# Load the latest version
tox21_df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "epicskills/tox21-dataset",
  file_path
)

#show(tox21_df)

# Valid molecules

In [ ]:
print("Dataset's shape",tox21_df.shape)
target_cols=[col for col in tox21_df.columns if col.startswith(('NR-', 'SR-'))]
n_colums_test=tox21_df.shape[1]-2
index_col=[i for i in range(0,n_colums_test)]
cmap = plt.get_cmap("nipy_spectral") 
colors = [cmap(i) for i in np.linspace(0.05, 0.95,n_colums_test)]

shape of dataset (7831, 14)


In [ ]:
valid_molecule=tox21_df.shape[0]-tox21_df[target_cols].isnull().sum()

plt.figure(figsize=(12, 7))
plt.bar(target_cols,valid_molecule,color=colors,edgecolor='black', linewidth=0.5)
plt.xticks(rotation=30, ha='right', fontsize=12)
plt.title('molecole valide per ogni test biologico (Tox21)', fontsize=15, pad=20)
plt.xlabel('dati validi su test biologici', fontsize=12)
plt.ylabel('Test Biologico', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.3)

for i,v in enumerate(valid_molecule.values):
    plt.text(i, v , f'{v}', color='black', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
Nan_values=tox21_df[target_cols].isnull().sum()

plt.figure(figsize=(12, 7))
 
plt.bar(target_cols,Nan_values,color=colors,edgecolor='black', linewidth=0.5)
plt.xticks(rotation=30, ha='right', fontsize=12)
plt.title('Valori Nulli per ogni test biologico (Tox21)', fontsize=15, pad=20)
plt.xlabel('dati mancanti', fontsize=12)
plt.ylabel('Test Biologico', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.3)


for i,v in enumerate(Nan_values.values):
    plt.text(i, v , f'{v}', color='black', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


# Class Imbalance

In [ ]:
melted_tox21=tox21_df[target_cols].melt(var_name="Test",value_name="Esito")
df_melted = melted_tox21.dropna()
plt.figure(figsize=(10, 10))

ax= sns.countplot(data=df_melted, x='Test', hue='Esito', palette={0.0: "#2ecc71", 1.0: "#fc1900"})
for container in ax.containers:
    ax.bar_label(container, padding=3, fontsize=9, fontweight='bold')
    
plt.xticks(rotation=30, ha='right')
plt.title('Sbilanciamento delle Classi: Molecole Sane (0) vs Tossiche (1)', fontsize=16)
plt.xlabel('Test Biologico', fontsize=12)
plt.ylabel('Numero di Molecole', fontsize=12)
plt.legend(title='Legenda', labels=['Sano (0)', 'Tossico (1)'])

plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Tests' correlation

In [ ]:
plt.figure(figsize=(10, 8))
corr = tox21_df[target_cols].corr()

sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlazione tra i 12 Test Biologici')
plt.xticks(rotation=45, ha='right')
plt.show()

# LogP analysis
--- 



In [12]:
def get_mw(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return Descriptors.MolWt(mol) if mol else None

def get_logp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return Descriptors.MolLogP(mol) if mol else None

In [ ]:
tox21_df['MW'] = tox21_df['smiles'].apply(get_mw)
tox21_df['LogP'] = tox21_df['smiles'].apply(get_logp)


tox21_df = tox21_df.dropna(subset=['MW', 'LogP'])
target_cols = [c for c in tox21_df.columns if c.startswith(('NR-', 'SR-'))]


tox21_df['Global_Tox'] = tox21_df[target_cols].max(axis=1).astype(int).astype(str) #Serve a creare un'unica etichetta che riassume se una molecola è pericolosa oppure no, indipendentemente dal test specifico.

# Grafico
plt.figure(figsize=(12, 6))
sns.violinplot(data=tox21_df, x='Global_Tox', y='LogP', palette={'0': "#2ecc71", '1': "#e74c3c"},inner="quartile")

plt.title('Distribuzione del LogP: Molecole Sane vs Tossiche', fontsize=15)
plt.xlabel('Tossicità Globale (0=Sano, 1=Tossico)')
plt.ylabel('LogP (Lipofilia)')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Molecular weight

In [ ]:
plt.figure(figsize=(12, 6))
sns.violinplot(data=tox21_df, x='Global_Tox', y='MW', palette={'0': "#2ecc71", '1': "#e74c3c"},inner="quartile")

plt.title('Distribuzione del MW: Molecole Sane vs Tossiche', fontsize=15)
plt.xlabel('Tossicità Globale (0=Sano, 1=Tossico)')
plt.ylabel('MW')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Test of toxicity for each molecule

In [ ]:

num_molecules_toxicity=[]
for i in range(0,len(index_col)-3):
    num_tox=tox21_df[target_cols[i]]== 1.0
    num_molecules_toxicity.append((num_tox==True).sum())

plt.figure(figsize=(12,8))
plt.bar(target_cols,num_molecules_toxicity,color=colors,edgecolor='black' ,linewidth=0.5)
plt.title('Tossicità di ogni molecola su 12 test', fontsize=15)
plt.xlabel('Test biologici')
plt.ylabel('Numnero di molecole')
plt.xticks(rotation=30, ha='right')

for i,v in enumerate(num_molecules_toxicity):
    plt.text(i, v , f'{v}', color='black', ha='center', va='bottom', fontweight='bold')

plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


# Molecular Space in 3D to highlight cluster of tossicity


In [ ]:
def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        # Generiamo il Morgan Fingerprint (raggio 2 = ECFP4)
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
        return np.array(fp)
    return None

# Trasformation in Morgan Fingerprints 

In [ ]:
fps = np.array([smiles_to_fp(s) for s in tox21_df['smiles'] if smiles_to_fp(s) is not None])
labels = tox21_df['Global_Tox'].astype(int).values #global toxicity based at least 1 test equals to 1

In [ ]:
tsne = TSNE(n_components=3, random_state=42, perplexity=30)
X_3d = tsne.fit_transform(fps)

In [ ]:
df_plot = pd.DataFrame(X_3d, columns=['TSNE1', 'TSNE2', 'TSNE3'])
df_plot['Tossicità'] = labels.astype(str) # String to identify toxicity
df_plot['ID'] = tox21_df['mol_id'].values # Aggiungiamo l'ID per il mouse

# 2. Creiamo il grafico interattivo
fig = px.scatter_3d(
    df_plot, 
    x='TSNE1', y='TSNE2', z='TSNE3',
    color='Tossicità',
    color_discrete_map={'0': '#2ecc71', '1': '#e74c3c'},
    title="Spazio Chimico Tox21 - t-SNE 3D",
    hover_data=['ID'], # Mostra l'ID quando passi il mouse
    opacity=0.6
)

# 3. Rendiamo i punti più piccoli per migliorare la fluidità
fig.update_traces(marker=dict(size=2))
#fig.update_layout(scene=dict(xaxis=dict(visible=False),
 #                            yaxis=dict(visible=False),
 #                            zaxis=dict(visible=False)))

fig.show()

In [16]:
show(tox21_df)
mean_mw = tox21_df['MW'].mean()
std_mw = tox21_df['MW'].std()
mean_logp = tox21_df['LogP'].mean()
std_logp = tox21_df['LogP'].std()

print(f"MW: Mean={mean_mw}, Std={std_mw}")
print(f"LogP: Mean={mean_logp}, Std={std_logp}")

Loading ITables v2.6.2 from the internet... (need help?)


MW: Mean=276.1441553893223, Std=164.73235552463632
LogP: Mean=2.373942812220377, Std=2.3043073032948307


# Enriching Dataset

In [ ]:
columns_to_save = ['smiles', 'MW', 'LogP'] + target_cols
subset_df = tox21_df[columns_to_save]

# 4. (Opzionale) Rimuoviamo righe dove SMILES non è valido (MW è None/NaN)
initial_len = len(subset_df)
subset_df = subset_df.dropna(subset=['MW', 'LogP'])
print(f"Rimosse {initial_len - len(subset_df)} righe con SMILES non validi.")

# 5. Salvataggio su CSV
filename = DATASET_PATH
subset_df.to_csv(filename, index=False)

print(f"File salvato con successo: {filename}")
print(f"Dimensioni del file salvato: {subset_df.shape}")
print("-" * 30)
#print(subset_df.head())

# Show chemcical structure

In [ ]:
def compare_molecules_3d(smiles_toxic, smiles_safe, name_toxic="Tossica", name_safe="Non Tossica"):
    """
    Visualizza due molecole 3D affiancate per confronto.
    """
    # --- 1. Preparazione Molecola Tossica ---
    mol_tox = Chem.MolFromSmiles(smiles_toxic)
    mol_tox = Chem.AddHs(mol_tox)
    AllChem.EmbedMolecule(mol_tox, AllChem.ETKDG())
    AllChem.MMFFOptimizeMolecule(mol_tox)
    block_tox = Chem.MolToMolBlock(mol_tox)

    # --- 2. Preparazione Molecola Sicura ---
    mol_safe = Chem.MolFromSmiles(smiles_safe)
    mol_safe = Chem.AddHs(mol_safe)
    AllChem.EmbedMolecule(mol_safe, AllChem.ETKDG())
    AllChem.MMFFOptimizeMolecule(mol_safe)
    block_safe = Chem.MolToMolBlock(mol_safe)

    # --- 3. Setup della Vista Affiancata (Grid 1x2) ---
    # viewergrid=(righe, colonne) -> Qui facciamo 1 riga, 2 colonne
    view = py3Dmol.view(width=900, height=500, viewergrid=(1, 2))

    # --- 4. Aggiunta Modelli ai Pannelli ---
    # viewer=(0,0) -> Pannello Sinistro (Tossica)
    view.addModel(block_tox, 'mol', viewer=(0, 0))
    view.setStyle({'stick': {'colorscheme': 'redCarbon'}}, viewer=(0, 0)) # Carbonio Rosso
    view.addLabel(name_toxic, 
                  {'position': {'x': -2, 'y': 5, 'z': 0}, 'backgroundColor': "#b93e3e", 'fontColor': 'black'},
                  viewer=(0, 0))

    # viewer=(0,1) -> Pannello Destro (Sicura)
    view.addModel(block_safe, 'mol', viewer=(0, 1))
    view.setStyle({'stick': {'colorscheme': 'greenCarbon'}}, viewer=(0, 1)) # Carbonio Verde
    view.addLabel(name_safe, 
                  {'position': {'x': -2, 'y': 5, 'z': 0}, 'backgroundColor': "#6de26d", 'fontColor': 'black'},
                  viewer=(0, 1))

    # --- 5. Render Finale ---
    view.zoomTo() # Centra entrambe le camere
    view.show()

# ==========================================
# ESEMPIO CONCRETO
# ==========================================

# 1. Parathion (Pesticida altamente tossico - Tox21 Positive)
# Notare il gruppo P=S e il gruppo Nitro (NO2)
toxic_smiles = tox21_df["smiles"][0] #ottanoato di zinco

# 2. Aspirina (Farmaco sicuro - Tox21 Negative)
safe_smiles = tox21_df["smiles"][9] 
#safe_smiles="NCCc1ccc(O)c(O)c1"
print("Confronto: tossica(sinistra) vs non tossica(destra)")

compare_molecules_3d(toxic_smiles, safe_smiles, name_toxic="l'Ethoxzolamide", name_safe="Ottanoato di zinco")

Confronto: tossica(sinistra) vs non tossica(destra)


[14:53:29] UFFTYPER: Unrecognized charge state for atom: 0
[14:53:29] UFFTYPER: Unrecognized atom type: Zn+2 (0)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.